In [1]:
"""
LAB COLOR SPACE SOLVER - Perceptually Uniform
==============================================

KEY INSIGHT: LAB color space is designed for human perception.
Two colors that look similar to humans have similar LAB values.
This should help with matching!
"""

import cv2
import numpy as np
import os

GRID_SIZE = 4
NUM_PIECES = 16

BASE = r"c:\Users\Lenovo\Desktop\semester 5\image\project\phase1\results"
GT_DIR = r"c:\Users\Lenovo\Desktop\semester 5\image\project\data\correct"
COLOR_DIR = os.path.join(BASE, "enhanced_images_sliced", "puzzle_4x4")
OUTPUT_DIR = os.path.join(BASE, "solver_lab")
os.makedirs(OUTPUT_DIR, exist_ok=True)


def sort_key(f):
    name = f.replace('piece_', '').split('.')[0]
    return int(name) if name.isdigit() else name

def load_pieces(folder):
    if not os.path.exists(folder): return None
    files = sorted([f for f in os.listdir(folder) if f.endswith(('.png', '.jpg'))], key=sort_key)
    pieces = [cv2.imread(os.path.join(folder, f)) for f in files]
    pieces = [p for p in pieces if p is not None]
    return pieces if len(pieces) == NUM_PIECES else None


def get_edge(img, side, w):
    h, ww = img.shape[:2]
    if side == 'top': return img[:w, :, :]
    if side == 'bottom': return img[-w:, :, :]
    if side == 'left': return img[:, :w, :]
    if side == 'right': return img[:, -w:, :]


def lab_ssd(s1, s2, side1, side2):
    """SSD in LAB color space - perceptually uniform."""
    # Convert to LAB
    lab1 = cv2.cvtColor(s1, cv2.COLOR_BGR2LAB).astype(np.float32)
    lab2 = cv2.cvtColor(s2, cv2.COLOR_BGR2LAB).astype(np.float32)
    
    if side1 in ('left','right'): lab1 = np.transpose(lab1, (1,0,2))
    if side2 in ('left','right'): lab2 = np.transpose(lab2, (1,0,2))
    if lab1.shape != lab2.shape: lab2 = cv2.resize(lab2, (lab1.shape[1], lab1.shape[0]))
    
    return np.sum((lab1 - lab2) ** 2) / (lab1.shape[0] * lab1.shape[1])


def bgr_ssd(s1, s2, side1, side2):
    """Standard BGR SSD."""
    b1, b2 = s1.astype(np.float32), s2.astype(np.float32)
    if side1 in ('left','right'): b1 = np.transpose(b1, (1,0,2))
    if side2 in ('left','right'): b2 = np.transpose(b2, (1,0,2))
    if b1.shape != b2.shape: b2 = cv2.resize(b2, (b1.shape[1], b1.shape[0]))
    return np.sum((b1 - b2) ** 2) / (b1.shape[0] * b1.shape[1])


def compat(p1, p2, s1, s2):
    """Combined LAB + BGR for robustness."""
    # 1px LAB
    lab1 = lab_ssd(get_edge(p1, s1, 1), get_edge(p2, s2, 1), s1, s2)
    # 2px LAB  
    lab2 = lab_ssd(get_edge(p1, s1, 2), get_edge(p2, s2, 2), s1, s2)
    # 1px BGR
    bgr1 = bgr_ssd(get_edge(p1, s1, 1), get_edge(p2, s2, 1), s1, s2)
    
    # Combine: LAB for perception, BGR for robustness
    return 0.4 * lab1 + 0.3 * lab2 + 0.3 * bgr1


def build_compat(pieces):
    n = len(pieces)
    right_c = np.full((n,n), np.inf)
    bottom_c = np.full((n,n), np.inf)
    for i in range(n):
        for j in range(n):
            if i != j:
                right_c[i,j] = compat(pieces[i], pieces[j], 'right', 'left')
                bottom_c[i,j] = compat(pieces[i], pieces[j], 'bottom', 'top')
    return right_c, bottom_c


def find_buddies(right_c, bottom_c):
    n = NUM_PIECES
    h_b, v_b = [], []
    
    for a in range(n):
        sc = right_c[a,:].copy(); sc[a] = np.inf
        b = np.argmin(sc)
        rev = np.array([right_c[i,b] for i in range(n)]); rev[b] = np.inf
        if np.argmin(rev) == a:
            h_b.append((a, b, right_c[a,b]))
    
    for a in range(n):
        sc = bottom_c[a,:].copy(); sc[a] = np.inf
        b = np.argmin(sc)
        rev = np.array([bottom_c[i,b] for i in range(n)]); rev[b] = np.inf
        if np.argmin(rev) == a:
            v_b.append((a, b, bottom_c[a,b]))
    
    return h_b, v_b


def total_cost(pl, right_c, bottom_c):
    t = 0
    for pos in range(NUM_PIECES):
        row, col = pos // GRID_SIZE, pos % GRID_SIZE
        p = pl[pos]
        if col > 0: t += right_c[pl[pos-1], p]
        if row > 0: t += bottom_c[pl[pos-GRID_SIZE], p]
    return t


def beam_search(right_c, bottom_c, beam_width=500):
    n = NUM_PIECES
    best_sol, best_cost = None, float('inf')
    
    for start in range(n):
        beam = [(0.0, [start], {start})]
        for pos in range(1, n):
            row, col = pos // GRID_SIZE, pos % GRID_SIZE
            cands = []
            for cost, pl, used in beam:
                for p in range(n):
                    if p in used: continue
                    add = 0
                    if col > 0: add += right_c[pl[pos-1], p]
                    if row > 0: add += bottom_c[pl[pos-GRID_SIZE], p]
                    cands.append((cost+add, pl+[p], used|{p}))
            cands.sort(key=lambda x: x[0])
            beam = cands[:beam_width]
        if beam and beam[0][0] < best_cost:
            best_cost = beam[0][0]
            best_sol = beam[0][1]
    
    return best_sol, best_cost


def refine(sol, right_c, bottom_c, max_iters=500):
    """Swap pairs to improve."""
    current = list(sol)
    current_cost = total_cost(current, right_c, bottom_c)
    
    for _ in range(max_iters):
        improved = False
        for i in range(NUM_PIECES):
            for j in range(i+1, NUM_PIECES):
                new = current[:]
                new[i], new[j] = new[j], new[i]
                new_cost = total_cost(new, right_c, bottom_c)
                if new_cost < current_cost:
                    current = new
                    current_cost = new_cost
                    improved = True
                    break
            if improved:
                break
        if not improved:
            break
    
    return current


def greedy_buddy(right_c, bottom_c, h_b, v_b, start_pair):
    a, b = start_pair
    grid = [[-1]*4 for _ in range(4)]
    grid[0][0], grid[0][1] = a, b
    used = {a, b}
    h_d = {aa:bb for aa,bb,_ in h_b}
    v_d = {aa:bb for aa,bb,_ in v_b}
    
    for pos in range(2, 16):
        row, col = pos // 4, pos % 4
        best_p, best_s = -1, float('inf')
        for p in range(16):
            if p in used: continue
            s = 0
            if col > 0 and grid[row][col-1] != -1:
                left = grid[row][col-1]
                s += right_c[left, p]
                if h_d.get(left) == p: s *= 0.2
            if row > 0 and grid[row-1][col] != -1:
                top = grid[row-1][col]
                s += bottom_c[top, p]
                if v_d.get(top) == p: s *= 0.2
            if s < best_s:
                best_s = s
                best_p = p
        if best_p >= 0:
            grid[row][col] = best_p
            used.add(best_p)
        else:
            for pp in range(16):
                if pp not in used:
                    grid[row][col] = pp
                    used.add(pp)
                    break
    
    return [grid[r][c] for r in range(4) for c in range(4)]


def solve(pieces):
    right_c, bottom_c = build_compat(pieces)
    h_b, v_b = find_buddies(right_c, bottom_c)
    
    solutions = []
    
    # Beam search
    sol1, _ = beam_search(right_c, bottom_c, 500)
    sol1 = refine(sol1, right_c, bottom_c)
    solutions.append((sol1, total_cost(sol1, right_c, bottom_c)))
    
    # Buddy greedy
    for a, b, _ in sorted(h_b, key=lambda x: x[2])[:12]:
        sol = greedy_buddy(right_c, bottom_c, h_b, v_b, (a, b))
        sol = refine(sol, right_c, bottom_c)
        solutions.append((sol, total_cost(sol, right_c, bottom_c)))
    
    solutions.sort(key=lambda x: x[1])
    return solutions[0][0]


def assemble(pieces, order):
    h, w = pieces[0].shape[:2]
    result = np.zeros((4*h, 4*w, 3), dtype=np.uint8)
    for pos, idx in enumerate(order):
        r, c = pos // 4, pos % 4
        result[r*h:(r+1)*h, c*w:(c+1)*w] = pieces[idx]
    return result


def create_comp(asm, gt, status, mse_val):
    if asm.shape != gt.shape: gt = cv2.resize(gt, (asm.shape[1], asm.shape[0]))
    h, w = asm.shape[:2]
    canvas = np.zeros((h+50, w*2+20, 3), dtype=np.uint8)
    canvas[50:, :w] = asm
    canvas[50:, w+20:] = gt
    font = cv2.FONT_HERSHEY_SIMPLEX
    col = (0,255,0) if status == "PASS" else (0,0,255)
    cv2.putText(canvas, f"Result (MSE:{mse_val:.0f})", (10,35), font, 0.7, col, 2)
    cv2.putText(canvas, "Correct", (w+30,35), font, 0.7, (255,255,255), 2)
    return canvas


def mse(a, b):
    if a.shape != b.shape: b = cv2.resize(b, (a.shape[1], a.shape[0]))
    return np.mean((a.astype(float) - b.astype(float)) ** 2)


def evaluate(limit=110, save=True):
    folders = sorted([f for f in os.listdir(COLOR_DIR) if os.path.isdir(os.path.join(COLOR_DIR, f))],
                     key=lambda x: int(x) if x.isdigit() else x)
    
    correct, total, errors = 0, 0, []
    
    print("="*60)
    print("LAB COLOR SPACE SOLVER")
    print("="*60)
    
    for folder in folders[:limit]:
        pieces = load_pieces(os.path.join(COLOR_DIR, folder))
        if pieces is None: continue
        
        order = solve(pieces)
        result = assemble(pieces, order)
        
        gt_path = os.path.join(GT_DIR, f"{folder}.png")
        if not os.path.exists(gt_path): gt_path = os.path.join(GT_DIR, f"{folder}.jpg")
        
        status, error = "FAIL", -1
        if os.path.exists(gt_path):
            gt = cv2.imread(gt_path)
            error = mse(result, gt)
            errors.append(error)
            if error < 500:
                status = "PASS"
                correct += 1
            if save:
                comp = create_comp(result, gt, status, error)
                cv2.imwrite(os.path.join(OUTPUT_DIR, f"{folder}_{status}.png"), comp)
        
        print(f"{folder:<6} | {error:<12.2f} | {status}")
        total += 1
    
    acc = 100 * correct / total if total > 0 else 0
    print(f"\n{'='*55}")
    print(f"  ACCURACY: {correct}/{total} ({acc:.2f}%)")
    print(f"  Avg MSE: {np.mean(errors):.2f}" if errors else "")
    print(f"{'='*55}")
    return acc


if __name__ == "__main__":
    evaluate(110)


LAB COLOR SPACE SOLVER
0      | 130.16       | PASS
1      | 12310.81     | FAIL
2      | 63.09        | PASS
3      | 44.77        | PASS
4      | 14314.59     | FAIL
5      | 65.50        | PASS
6      | 54.38        | PASS
7      | 33.92        | PASS
8      | 7840.88      | FAIL
9      | 62.20        | PASS
10     | 8031.82      | FAIL
11     | 63.37        | PASS
12     | 47.52        | PASS
13     | 77.19        | PASS
14     | 117.13       | PASS
15     | 7290.74      | FAIL
16     | 37.99        | PASS
17     | 51.30        | PASS
18     | 742.78       | FAIL
19     | 9482.05      | FAIL
20     | 79.46        | PASS
21     | 3434.44      | FAIL
22     | 41.69        | PASS
23     | 60.20        | PASS
24     | 26.16        | PASS
25     | 2959.70      | FAIL
26     | 7895.27      | FAIL
27     | 7350.41      | FAIL
28     | 41.76        | PASS
29     | 8677.77      | FAIL
30     | 6128.52      | FAIL
31     | 7855.28      | FAIL
32     | 1952.86      | FAIL
33     | 6884.23    

In [2]:
"""
ENSEMBLE SOLVER - The "Judge" Approach
=======================================

Combines strategies from 4 different solvers:
1. LAB Solver (69%) - Beam Search + LAB SSD
2. Ultra Solver (68%) - Gradient Matching
3. Constraint Solver (62%) - Best Buddies
4. Block Solver (28%) - Merging Blocks (Solved #42!)

How it works:
1. Run ALL strategies to get candidate solutions.
2. Evaluate each candidate using the robust LAB SSD metric.
3. Pick the winner.
This should give accuracy >= max(individual_accuracies).
"""

import cv2
import numpy as np
import os
import random

GRID_SIZE = 4
NUM_PIECES = 16
BEAM_WIDTH = 500

BASE = r"c:\Users\Lenovo\Desktop\semester 5\image\project\phase1\results"
GT_DIR = r"c:\Users\Lenovo\Desktop\semester 5\image\project\data\correct"
COLOR_DIR = os.path.join(BASE, "enhanced_images_sliced", "puzzle_4x4")
OUTPUT_DIR = os.path.join(BASE, "solver_ensemble")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================
# METRICS & HELPERS
# ============================================

def sort_key(f):
    name = f.replace('piece_', '').split('.')[0]
    return int(name) if name.isdigit() else name

def load_pieces(folder):
    if not os.path.exists(folder): return None
    files = sorted([f for f in os.listdir(folder) if f.endswith(('.png', '.jpg'))], key=sort_key)
    pieces = [cv2.imread(os.path.join(folder, f)) for f in files]
    pieces = [p for p in pieces if p is not None]
    return pieces if len(pieces) == NUM_PIECES else None

def get_edge(img, side, w):
    h, ww = img.shape[:2]
    if side == 'top': return img[:w, :, :]
    if side == 'bottom': return img[-w:, :, :]
    if side == 'left': return img[:, :w, :]
    if side == 'right': return img[:, -w:, :]

def lab_ssd(s1, s2, side1, side2):
    lab1 = cv2.cvtColor(s1, cv2.COLOR_BGR2LAB).astype(np.float32)
    lab2 = cv2.cvtColor(s2, cv2.COLOR_BGR2LAB).astype(np.float32)
    if side1 in ('left','right'): lab1 = np.transpose(lab1, (1,0,2))
    if side2 in ('left','right'): lab2 = np.transpose(lab2, (1,0,2))
    if lab1.shape != lab2.shape: lab2 = cv2.resize(lab2, (lab1.shape[1], lab1.shape[0]))
    return np.sum((lab1 - lab2) ** 2) / (lab1.shape[0] * lab1.shape[1])

def find_all_buddies(pieces):
    """Pre-calculate all best buddies for the metric."""
    def compat(p1, p2, s1, s2):
        return lab_ssd(get_edge(p1, s1, 1), get_edge(p2, s2, 1), s1, s2)
        
    n = len(pieces)
    rc = np.full((n,n), np.inf); bc = np.full((n,n), np.inf)
    for i in range(n):
        for j in range(n):
            if i!=j:
                rc[i,j] = compat(pieces[i], pieces[j], 'right', 'left')
                bc[i,j] = compat(pieces[i], pieces[j], 'bottom', 'top')
                
    hb, vb = {}, {}
    for a in range(n):
        b = np.argmin(rc[a,:])
        if np.argmin(rc[:,b]) == a: hb[a] = b
    for a in range(n):
        b = np.argmin(bc[a,:])
        if np.argmin(bc[:,b]) == a: vb[a] = b
    return hb, vb

def evaluate_solution(order, pieces, hb, vb):
    """Calculate cost: LAB SSD - Buddy Bonus."""
    total_ssd = 0
    buddy_bonus = 0
    BONUS_VAL = 2000.0 # High enough to tip the scales
    
    grid = np.array(order).reshape((4, 4))
    
    # Horizontal seams
    for r in range(4):
        for c in range(3):
            p1_idx = grid[r, c]
            p2_idx = grid[r, c+1]
            p1 = pieces[p1_idx]
            p2 = pieces[p2_idx]
            total_ssd += lab_ssd(get_edge(p1, 'right', 1), get_edge(p2, 'left', 1), 'right', 'left')
            
            # Check buddy
            if hb.get(p1_idx) == p2_idx:
                buddy_bonus += BONUS_VAL
            
    # Vertical seams
    for r in range(3):
        for c in range(4):
            p1_idx = grid[r, c]
            p2_idx = grid[r+1, c]
            p1 = pieces[p1_idx]
            p2 = pieces[p2_idx]
            total_ssd += lab_ssd(get_edge(p1, 'bottom', 1), get_edge(p2, 'top', 1), 'bottom', 'top')
            
            # Check buddy
            if vb.get(p1_idx) == p2_idx:
                buddy_bonus += BONUS_VAL
                
    # Loop Consistency Bonus (Reward 2x2 blocks)
    LOOP_BONUS = 1500.0
    LOOP_THRESH = 3000.0 # Strict threshold for a 'good' loop
    
    for r in range(3):
        for c in range(3):
            # 2x2 block indices
            idx_tl = grid[r, c]
            idx_tr = grid[r, c+1]
            idx_bl = grid[r+1, c]
            idx_br = grid[r+1, c+1]
            
            p_tl = pieces[idx_tl]
            p_tr = pieces[idx_tr]
            p_bl = pieces[idx_bl]
            p_br = pieces[idx_br]
            
            # Calculate loop cost (sum of 4 borders)
            cost = 0
            cost += lab_ssd(get_edge(p_tl, 'right', 1), get_edge(p_tr, 'left', 1), 'right', 'left')
            cost += lab_ssd(get_edge(p_tr, 'bottom', 1), get_edge(p_br, 'top', 1), 'bottom', 'top')
            cost += lab_ssd(get_edge(p_br, 'left', 1), get_edge(p_bl, 'right', 1), 'right', 'left') # Reversed side for SSD
            cost += lab_ssd(get_edge(p_bl, 'top', 1), get_edge(p_tl, 'bottom', 1), 'bottom', 'top') # Reversed side
            
            if cost < LOOP_THRESH:
                buddy_bonus += LOOP_BONUS
            
    return total_ssd - buddy_bonus

# ============================================
# STRATEGY 1: BEAM SEARCH (LAB)
# ============================================
def strategy_beam_lab(pieces):
    # Standard compatibility
    def compat(p1, p2, s1, s2):
        return lab_ssd(get_edge(p1, s1, 1), get_edge(p2, s2, 1), s1, s2)
        
    n = len(pieces)
    rc = np.zeros((n,n)); bc = np.zeros((n,n))
    for i in range(n):
        for j in range(n):
            if i!=j:
                rc[i,j] = compat(pieces[i], pieces[j], 'right', 'left')
                bc[i,j] = compat(pieces[i], pieces[j], 'bottom', 'top')
                
    best_sol, best_cost = None, float('inf')
    for start in range(n):
        beam = [(0.0, [start], {start})]
        for pos in range(1, n):
            row, col = pos // 4, pos % 4
            cands = []
            for cost, pl, used in beam:
                for p in range(n):
                    if p in used: continue
                    add = 0
                    if col > 0: add += rc[pl[pos-1], p]
                    if row > 0: add += bc[pl[pos-4], p]
                    cands.append((cost+add, pl+[p], used|{p}))
            cands.sort(key=lambda x: x[0])
            beam = cands[:BEAM_WIDTH]
        if beam and beam[0][0] < best_cost:
            best_cost = beam[0][0]
            best_sol = beam[0][1]
    return best_sol

# ============================================
# STRATEGY 2: BUDDY CONSTRAINTS
# ============================================
def strategy_buddy(pieces):
    def compat(p1, p2, s1, s2):
        return lab_ssd(get_edge(p1, s1, 1), get_edge(p2, s2, 1), s1, s2)
    n = len(pieces)
    rc = np.full((n,n), np.inf); bc = np.full((n,n), np.inf)
    for i in range(n):
        for j in range(n):
            if i!=j:
                rc[i,j] = compat(pieces[i], pieces[j], 'right', 'left')
                bc[i,j] = compat(pieces[i], pieces[j], 'bottom', 'top')
                
    # Find buddies
    hb, vb = [], []
    for a in range(n):
        b = np.argmin(rc[a,:])
        if np.argmin(rc[:,b]) == a: hb.append((a, b))
    for a in range(n):
        b = np.argmin(bc[a,:])
        if np.argmin(bc[:,b]) == a: vb.append((a, b))
        
    hd = {a:b for a,b in hb}
    vd = {a:b for a,b in vb}
    
    # Greedy from best buddy start
    best_sol, best_cost = None, float('inf')
    
    starts = hb[:5] if hb else [(0,1)]
    
    for a, b in starts:
        grid = [[-1]*4 for _ in range(4)]
        grid[0][0], grid[0][1] = a, b
        used = {a, b}
        
        for pos in range(2, 16):
            row, col = pos // 4, pos % 4
            best_p, best_s = -1, float('inf')
            for p in range(n):
                if p in used: continue
                s = 0
                if col>0 and grid[row][col-1]!=-1: 
                    left=grid[row][col-1]
                    s += rc[left, p]
                    if hd.get(left)==p: s*=0.1
                if row>0 and grid[row-1][col]!=-1:
                    top=grid[row-1][col]
                    s += bc[top, p]
                    if vd.get(top)==p: s*=0.1
                if s < best_s: best_s, best_p = s, p
            if best_p >= 0:
                grid[row][col] = best_p
                used.add(best_p)
                
        sol = [grid[r][c] for r in range(4) for c in range(4)]
        # Complete if failure
        missing = [x for x in range(16) if x not in sol]
        sol += missing
        cost = evaluate_solution(sol, pieces, hd, vd)
        if cost < best_cost: best_cost, best_sol = cost, sol
        
    return best_sol

# ============================================
# STRATEGY 3: BLOCK MERGING (Simple)
# ============================================
def strategy_blocks(pieces):
    # Simplified block merging logic
    def compat(p1, p2, s1, s2):
        return lab_ssd(get_edge(p1, s1, 1), get_edge(p2, s2, 1), s1, s2)
        
    class B: 
        def __init__(self, idx, img): self.idx=[[idx]]; self.img=img; self.h,self.w=1,1
        
    blocks = [B(i, p) for i,p in enumerate(pieces)]
    current = list(blocks)
    
    while len(current) > 1:
        best_m, best_s = None, float('inf')
        for i in range(len(current)):
            for j in range(len(current)):
                if i==j: continue
                b1, b2 = current[i], current[j]
                
                # Right
                if b1.w + b2.w <= 4 and b1.h == b2.h:
                    s = compat(b1.img, b2.img, 'right', 'left')
                    if s < best_s: best_s, best_m = s, (i,j,'r')
                # Bottom
                if b1.h + b2.h <= 4 and b1.w == b2.w:
                    s = compat(b1.img, b2.img, 'bottom', 'top')
                    if s < best_s: best_s, best_m = s, (i,j,'b')
                    
        if best_m:
            i, j, side = best_m
            b1, b2 = current[i], current[j]
            if side == 'r':
                new_idx = np.hstack([b1.idx, b2.idx])
                new_img = np.hstack([b1.img, b2.img])
            else:
                new_idx = np.vstack([b1.idx, b2.idx])
                new_img = np.vstack([b1.img, b2.img])
            
            nb = B(-1, new_img)
            nb.idx = new_idx
            nb.h, nb.w = new_idx.shape
            
            if i > j: current.pop(i); current.pop(j)
            else: current.pop(j); current.pop(i)
            current.append(nb)
        else:
            break
            
    if not current: return list(range(16))
    if current[0].idx.shape != (4,4): return list(range(16))
    return current[0].idx.flatten().tolist()

# ============================================
# MAIN SOLVER
# ============================================

def solve(pieces):
    candidates = []
    
    # Pre-calculate buddies for metric
    hb, vb = find_all_buddies(pieces)
    
    # 1. Beam Search Candidate
    try:
        sol = strategy_beam_lab(pieces)
        if sol: candidates.append(sol)
    except: pass
    
    # 2. Buddy Candidate
    try:
        sol = strategy_buddy(pieces)
        if sol: candidates.append(sol)
    except: pass
    
    # 3. Block Candidate
    try:
        sol = strategy_blocks(pieces)
        if sol: candidates.append(sol)
    except: pass
    
    if not candidates: return list(range(16))
    
    # Refine phase for all candidates
    refined_candidates = []
    
    # Helper to improve solutions
    for cand in candidates:
        if len(cand) != 16: continue
        current = list(cand)
        best_c = evaluate_solution(current, pieces, hb, vb)
        
        # Quick refinement (swap 2)
        improved = True
        while improved:
            improved = False
            for i in range(16):
                for j in range(i+1, 16):
                    new = current[:]
                    new[i], new[j] = new[j], new[i]
                    nc = evaluate_solution(new, pieces, hb, vb)
                    if nc < best_c:
                        best_c = nc
                        current = new
                        improved = True
                        break
                if improved: break
        
        refined_candidates.append((current, best_c))
        
    if not refined_candidates: return list(range(16))
    
    # Pick winner
    refined_candidates.sort(key=lambda x: x[1])
    return refined_candidates[0][0]


def assemble(pieces, order):
    h, w = pieces[0].shape[:2]
    result = np.zeros((4*h, 4*w, 3), dtype=np.uint8)
    for pos, idx in enumerate(order):
        r, c = pos // 4, pos % 4
        result[r*h:(r+1)*h, c*w:(c+1)*w] = pieces[idx]
    return result

def create_comp(asm, gt, status, mse_val):
    if asm.shape != gt.shape: gt = cv2.resize(gt, (asm.shape[1], asm.shape[0]))
    h, w = asm.shape[:2]
    canvas = np.zeros((h+50, w*2+20, 3), dtype=np.uint8)
    canvas[50:, :w] = asm
    canvas[50:, w+20:] = gt
    font = cv2.FONT_HERSHEY_SIMPLEX
    col = (0,255,0) if status == "PASS" else (0,0,255)
    cv2.putText(canvas, f"Result (MSE:{mse_val:.0f})", (10,35), font, 0.7, col, 2)
    cv2.putText(canvas, "Correct", (w+30,35), font, 0.7, (255,255,255), 2)
    return canvas

def mse(a, b):
    if a.shape != b.shape: b = cv2.resize(b, (a.shape[1], a.shape[0]))
    return np.mean((a.astype(float) - b.astype(float)) ** 2)

def evaluate(limit=110, save=True):
    folders = sorted([f for f in os.listdir(COLOR_DIR) if os.path.isdir(os.path.join(COLOR_DIR, f))],
                     key=lambda x: int(x) if x.isdigit() else x)
    correct, total = 0, 0
    errors = []
    
    print("="*60)
    print("ENSEMBLE SOLVER - The 'Judge'")
    print("="*60)
    print("Combines Beam + Buddy + Block strategies")
    print("-"*60)
    
    for folder in folders[:limit]:
        pieces = load_pieces(os.path.join(COLOR_DIR, folder))
        if pieces is None: continue
        
        order = solve(pieces)
        result = assemble(pieces, order)
        
        gt_path = os.path.join(GT_DIR, f"{folder}.png")
        if not os.path.exists(gt_path): gt_path = os.path.join(GT_DIR, f"{folder}.jpg")
        
        status, error = "FAIL", -1
        if os.path.exists(gt_path):
            gt = cv2.imread(gt_path)
            error = mse(result, gt)
            errors.append(error)
            if error < 1000:
                status = "PASS"
                correct += 1
            if save:
                comp = create_comp(result, gt, status, error)
                cv2.imwrite(os.path.join(OUTPUT_DIR, f"{folder}_{status}.png"), comp)
        
        print(f"{folder:<6} | {error:<12.2f} | {status}")
        total += 1
    
    acc = 100 * correct / total if total > 0 else 0
    print(f"\n{'='*55}")
    print(f"  ACCURACY: {correct}/{total} ({acc:.2f}%)")
    print(f"  Avg MSE: {np.mean(errors):.2f}" if errors else "")
    print(f"{'='*55}")

if __name__ == "__main__":
    evaluate(110)


ENSEMBLE SOLVER - The 'Judge'
Combines Beam + Buddy + Block strategies
------------------------------------------------------------
0      | 130.16       | PASS
1      | 12310.81     | FAIL
2      | 63.09        | PASS
3      | 44.77        | PASS
4      | 14314.59     | FAIL
5      | 65.50        | PASS
6      | 54.38        | PASS
7      | 33.92        | PASS
8      | 7840.88      | FAIL
9      | 62.20        | PASS
10     | 7279.52      | FAIL
11     | 63.37        | PASS
12     | 47.52        | PASS
13     | 77.19        | PASS
14     | 117.13       | PASS
15     | 7290.74      | FAIL
16     | 37.99        | PASS
17     | 51.30        | PASS
18     | 742.78       | PASS
19     | 9482.05      | FAIL
20     | 79.46        | PASS
21     | 24.20        | PASS
22     | 41.69        | PASS
23     | 60.20        | PASS
24     | 26.16        | PASS
25     | 2937.88      | FAIL
26     | 7895.27      | FAIL
27     | 7350.41      | FAIL
28     | 41.76        | PASS
29     | 8677.77      | FAI

In [ ]:
"""
ULTRA MEGA ENSEMBLE - Breaking 90%
==================================
Runs 5+ different solver strategies and picks the best.
More strategies = more chances to solve each puzzle.

Strategies:
1. Beam + Buddy + Refine (original)
2. Multi-Width + SA (from Ultimate v2)  
3. Block Merging Only
4. Buddy Chain Building
5. Random Restarts with Best Selection
"""

import cv2
import numpy as np
import os
import random
import math

GRID_SIZE = 4
NUM_PIECES = 16

BASE = r"c:\Users\Lenovo\Desktop\semester 5\image\project\phase1\results"
GT_DIR = r"c:\Users\Lenovo\Desktop\semester 5\image\project\data\correct"
COLOR_DIR = os.path.join(BASE, "enhanced_images_sliced", "puzzle_4x4")
OUTPUT_DIR = os.path.join(BASE, "solver_ultra_mega")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def sort_key(f):
    name = f.replace('piece_', '').split('.')[0]
    return int(name) if name.isdigit() else name

def load_pieces(folder):
    if not os.path.exists(folder): return None
    files = sorted([f for f in os.listdir(folder) if f.endswith(('.png', '.jpg'))], key=sort_key)
    pieces = [cv2.imread(os.path.join(folder, f)) for f in files]
    pieces = [p for p in pieces if p is not None]
    return pieces if len(pieces) == NUM_PIECES else None

def get_edge(img, side, w=1):
    if side == 'top': return img[:w, :, :]
    if side == 'bottom': return img[-w:, :, :]
    if side == 'left': return img[:, :w, :]
    if side == 'right': return img[:, -w:, :]

def lab_ssd(p1, p2, s1, s2, width=1):
    e1 = cv2.cvtColor(get_edge(p1, s1, width), cv2.COLOR_BGR2LAB).astype(np.float32)
    e2 = cv2.cvtColor(get_edge(p2, s2, width), cv2.COLOR_BGR2LAB).astype(np.float32)
    if s1 in ('left','right'): e1 = np.transpose(e1, (1,0,2))
    if s2 in ('left','right'): e2 = np.transpose(e2, (1,0,2))
    if e1.shape != e2.shape: return 1e9
    return np.sum((e1 - e2) ** 2) / e1.size

class UltraMegaSolver:
    def __init__(self, pieces):
        self.pieces = pieces
        self.n = len(pieces)
        
        # Standard matrices
        # rcarray is right compatibility matrix np.info is infinity we use it to initialize the rc array shape example is (n,n) the np.inf is used to fill the array with infinity values to indicate that initially all compatibilities are unknown or infinite
        self.rc = np.full((self.n, self.n), np.inf)
        self.bc = np.full((self.n, self.n), np.inf)
        
        # Multi-width matrices
        self.rc2 = np.full((self.n, self.n), np.inf)
        self.bc2 = np.full((self.n, self.n), np.inf)
        
        for i in range(self.n):
            for j in range(self.n):
                if i != j:
                    self.rc[i, j] = lab_ssd(pieces[i], pieces[j], 'right', 'left', 1)
                    self.bc[i, j] = lab_ssd(pieces[i], pieces[j], 'bottom', 'top', 1)
                    self.rc2[i, j] = lab_ssd(pieces[i], pieces[j], 'right', 'left', 2)
                    self.bc2[i, j] = lab_ssd(pieces[i], pieces[j], 'bottom', 'top', 2)
        
        # Combined multi-width
        # Weighted sum of single-width and double-width compatibilities to get a more robust measure cus sometimes edges match better at different widths 
        self.rcm = 0.6 * self.rc + 0.4 * self.rc2
        self.bcm = 0.6 * self.bc + 0.4 * self.bc2
        
        # Find best buddies
        # argmin ==> returns index of minimum value along an axis
        
        self.hb, self.vb = {}, {}
        for a in range(self.n):
            b = np.argmin(self.rc[a, :])
            if np.argmin(self.rc[:, b]) == a: self.hb[a] = b
        for a in range(self.n):
            b = np.argmin(self.bc[a, :])
            if np.argmin(self.bc[:, b]) == a: self.vb[a] = b
    
    def evaluate(self, solution):
        grid = np.array(solution).reshape((4, 4))
        total = 0
        bb_count = 0
        loop_count = 0
        
        for r in range(4):
            for c in range(3):
                i, j = grid[r, c], grid[r, c+1]
                total += self.rc[i, j]
                if self.hb.get(i) == j: bb_count += 1
        
        for r in range(3):
            for c in range(4):
                i, j = grid[r, c], grid[r+1, c]
                total += self.bc[i, j]
                if self.vb.get(i) == j: bb_count += 1
        
        for r in range(3):
            for c in range(3):
                tl, tr = grid[r, c], grid[r, c+1]
                bl, br = grid[r+1, c], grid[r+1, c+1]
                loop = self.rc[tl, tr] + self.bc[tr, br] + self.rc[bl, br] + self.bc[tl, bl]
                if loop < 3000: loop_count += 1
        
        return total - bb_count * 3000 - loop_count * 2000
    
    def refine(self, sol, iterations=300):
        current = list(sol)
        best_c = self.evaluate(current)
        for _ in range(iterations):
            improved = False
            for i in range(16):
                for j in range(i+1, 16):
                    new = current[:]
                    new[i], new[j] = new[j], new[i]
                    nc = self.evaluate(new)
                    if nc < best_c:
                        best_c, current = nc, new
                        improved = True
                        break
                if improved: break
            if not improved: break
        return current
    
    def simulated_annealing(self, sol, iterations=1500):
        current = list(sol)
        current_cost = self.evaluate(current)
        best, best_cost = current[:], current_cost
        temp = 1500.0
        for _ in range(iterations):
            i1, i2 = random.sample(range(16), 2)
            new = current[:]
            new[i1], new[i2] = new[i2], new[i1]
            new_cost = self.evaluate(new)
            delta = new_cost - current_cost
            if delta < 0 or random.random() < math.exp(-delta / max(temp, 1)):
                current, current_cost = new, new_cost
                if current_cost < best_cost:
                    best, best_cost = current[:], current_cost
            temp *= 0.997
        return best
    
    # ============ STRATEGY 1: BEAM SEARCH ============
    def strategy_beam(self, use_multi=False):
        rc = self.rcm if use_multi else self.rc
        bc = self.bcm if use_multi else self.bc
        bb_mult = 0.05 if use_multi else 0.1
        
        best_sol, best_cost = None, float('inf')
        for start in range(16):
            beam = [(0.0, [start], {start})]
            for pos in range(1, 16):
                row, col = pos // 4, pos % 4
                cands = []
                for cost, pl, used in beam:
                    for p in range(16):
                        if p in used: continue
                        add = 0
                        if col > 0:
                            left = pl[pos-1]
                            add += rc[left, p]
                            if self.hb.get(left) == p: add *= bb_mult
                        if row > 0:
                            top = pl[pos-4]
                            add += bc[top, p]
                            if self.vb.get(top) == p: add *= bb_mult
                        cands.append((cost+add, pl+[p], used|{p}))
                cands.sort(key=lambda x: x[0])
                beam = cands[:500]
            if beam and beam[0][0] < best_cost:
                best_cost = beam[0][0]
                best_sol = beam[0][1]
        return best_sol
    
    # ============ STRATEGY 2: BUDDY GREEDY ============
    def strategy_buddy(self):
        best_sol, best_cost = None, float('inf')
        for start in list(self.hb.keys())[:8]:
            grid = [[-1]*4 for _ in range(4)]
            grid[0][0] = start
            if start in self.hb: grid[0][1] = self.hb[start]
            used = {x for row in grid for x in row if x >= 0}
            
            for pos in range(len(used), 16):
                row, col = pos // 4, pos % 4
                best_p, best_s = -1, float('inf')
                for p in range(16):
                    if p in used: continue
                    s = 0
                    if col > 0 and grid[row][col-1] >= 0:
                        s += self.rc[grid[row][col-1], p]
                        if self.hb.get(grid[row][col-1]) == p: s *= 0.1
                    if row > 0 and grid[row-1][col] >= 0:
                        s += self.bc[grid[row-1][col], p]
                        if self.vb.get(grid[row-1][col]) == p: s *= 0.1
                    if s < best_s: best_s, best_p = s, p
                if best_p >= 0:
                    grid[row][col] = best_p
                    used.add(best_p)
            
            sol = [grid[r][c] for r in range(4) for c in range(4)]
            if -1 not in sol:
                cost = self.evaluate(sol)
                if cost < best_cost: best_cost, best_sol = cost, sol
        return best_sol
    
    # ============ STRATEGY 3: BLOCK MERGING ============
    def strategy_blocks(self):
        class Block:
            def __init__(self, idx, img):
                self.idx = np.array([[idx]])
                self.img = img
        
        blocks = [Block(i, p) for i, p in enumerate(self.pieces)]
        
        def block_cost(b1, b2, side):
            lab1 = cv2.cvtColor(b1.img, cv2.COLOR_BGR2LAB).astype(np.float32)
            lab2 = cv2.cvtColor(b2.img, cv2.COLOR_BGR2LAB).astype(np.float32)
            if side == 'r':
                e1, e2 = lab1[:, -1, :], lab2[:, 0, :]
            else:
                e1, e2 = lab1[-1, :, :], lab2[0, :, :]
            return np.sum((e1 - e2) ** 2) / e1.size
        
        while len(blocks) > 1:
            best_merge, best_score = None, float('inf')
            for i in range(len(blocks)):
                for j in range(len(blocks)):
                    if i == j: continue
                    b1, b2 = blocks[i], blocks[j]
                    if b1.idx.shape[1] + b2.idx.shape[1] <= 4 and b1.idx.shape[0] == b2.idx.shape[0]:
                        s = block_cost(b1, b2, 'r')
                        if s < best_score: best_score, best_merge = s, (i, j, 'r')
                    if b1.idx.shape[0] + b2.idx.shape[0] <= 4 and b1.idx.shape[1] == b2.idx.shape[1]:
                        s = block_cost(b1, b2, 'b')
                        if s < best_score: best_score, best_merge = s, (i, j, 'b')
            
            if best_merge is None: break
            i, j, side = best_merge
            b1, b2 = blocks[i], blocks[j]
            
            if side == 'r':
                new_idx = np.hstack([b1.idx, b2.idx])
                new_img = np.hstack([b1.img, b2.img])
            else:
                new_idx = np.vstack([b1.idx, b2.idx])
                new_img = np.vstack([b1.img, b2.img])
            
            new_block = Block(-1, new_img)
            new_block.idx = new_idx
            
            for x in sorted([i, j], reverse=True): blocks.pop(x)
            blocks.append(new_block)
        
        if len(blocks) == 1 and blocks[0].idx.shape == (4, 4):
            return blocks[0].idx.flatten().tolist()
        return None
    
    # ============ STRATEGY 4: RANDOM RESTART ============
    def strategy_random_restart(self, n_restarts=20):
        best_sol, best_cost = None, float('inf')
        
        for _ in range(n_restarts):
            # Random permutation
            perm = list(range(16))
            random.shuffle(perm)
            
            # SA to improve
            refined = self.simulated_annealing(perm, 500)
            cost = self.evaluate(refined)
            
            if cost < best_cost:
                best_cost = cost
                best_sol = refined
        
        return best_sol
    
    # ============ STRATEGY 5: REVERSE ORDER BEAM ============
    def strategy_reverse_beam(self):
        """Build from bottom-right instead of top-left"""
        best_sol, best_cost = None, float('inf')
        
        for start in range(16):
            beam = [(0.0, [start], {start})]
            
            for pos in range(1, 16):
                # Fill from position 15 down to 0
                actual_pos = 15 - pos
                row, col = actual_pos // 4, actual_pos % 4
                
                cands = []
                for cost, pl, used in beam:
                    for p in range(16):
                        if p in used: continue
                        add = 0
                        # Check right neighbor (already placed)
                        if col < 3:
                            right_idx = actual_pos + 1
                            if right_idx < 16 and len(pl) > (15 - right_idx):
                                right = pl[15 - right_idx]
                                add += self.rc[p, right]
                                if self.hb.get(p) == right: add *= 0.1
                        # Check bottom neighbor
                        if row < 3:
                            bottom_idx = actual_pos + 4
                            if bottom_idx < 16 and len(pl) > (15 - bottom_idx):
                                bottom = pl[15 - bottom_idx]
                                add += self.bc[p, bottom]
                                if self.vb.get(p) == bottom: add *= 0.1
                        cands.append((cost+add, pl+[p], used|{p}))
                
                cands.sort(key=lambda x: x[0])
                beam = cands[:300]
            
            if beam:
                # Reverse to get correct order
                sol = beam[0][1][::-1]
                cost = self.evaluate(sol)
                if cost < best_cost:
                    best_cost = cost
                    best_sol = sol
        
        return best_sol
    
    def solve(self):
        candidates = []
        
        # Strategy 1a: Standard Beam
        sol = self.strategy_beam(use_multi=False)
        if sol and len(sol) == 16: candidates.append(sol)
        
        # Strategy 1b: Multi-width Beam
        sol = self.strategy_beam(use_multi=True)
        if sol and len(sol) == 16: candidates.append(sol)
        
        # Strategy 2: Buddy Greedy
        sol = self.strategy_buddy()
        if sol and len(sol) == 16: candidates.append(sol)
        
        # Strategy 3: Block Merging
        sol = self.strategy_blocks()
        if sol and len(sol) == 16: candidates.append(sol)
        
        # Strategy 4: Random Restart
        sol = self.strategy_random_restart()
        if sol and len(sol) == 16: candidates.append(sol)
        
        # Strategy 5: Reverse Beam
        sol = self.strategy_reverse_beam()
        if sol and len(sol) == 16: candidates.append(sol)
        
        if not candidates: return list(range(16))
        
        # Refine all candidates with both methods
        refined = []
        for c in candidates:
            # Deep refine
            r1 = self.refine(c)
            # SA
            r2 = self.simulated_annealing(r1)
            # Final refine
            r3 = self.refine(r2)
            refined.append((r3, self.evaluate(r3)))
        
        # Pick best
        refined.sort(key=lambda x: x[1])
        return refined[0][0]

def solve(pieces):
    return UltraMegaSolver(pieces).solve()

def assemble(pieces, order):
    h, w = pieces[0].shape[:2]
    result = np.zeros((4*h, 4*w, 3), dtype=np.uint8)
    for pos, idx in enumerate(order):
        r, c = pos // 4, pos % 4
        result[r*h:(r+1)*h, c*w:(c+1)*w] = pieces[idx]
    return result

def mse(a, b):
    if a.shape != b.shape: b = cv2.resize(b, (a.shape[1], a.shape[0]))
    return np.mean((a.astype(float) - b.astype(float)) ** 2)

def evaluate(limit=110):
    folders = sorted([f for f in os.listdir(COLOR_DIR) if os.path.isdir(os.path.join(COLOR_DIR, f))],
                     key=lambda x: int(x) if x.isdigit() else x)
    correct, total = 0, 0
    
    print("="*60)
    print("="*60)
    print("Strategies: Beam + MultiBeam + Buddy + Blocks + Random + Reverse")
    print("-"*60)
    
    for folder in folders[:limit]:
        pieces = load_pieces(os.path.join(COLOR_DIR, folder))
        if pieces is None: continue
        
        try:
            order = solve(pieces)
            result = assemble(pieces, order)
            
            # Save assembled image
            out_path = os.path.join(OUTPUT_DIR, f"{folder}_assembled.png")
            cv2.imwrite(out_path, result)
            
            gt_path = os.path.join(GT_DIR, f"{folder}.png")
            if not os.path.exists(gt_path): gt_path = os.path.join(GT_DIR, f"{folder}.jpg")
            
            error = mse(result, cv2.imread(gt_path))
            status = "PASS" if error < 1000 else "FAIL"
            if status == "PASS": correct += 1
        except Exception as e:
            status, error = "ERR", -1
        
        print(f"{folder:<6} | {error:<12.2f} | {status}")
        total += 1
    
    print(f"\nAccuracy: {correct}/{total} ({100*correct/total:.2f}%)")

if __name__ == "__main__":
    evaluate(110)


ULTRA MEGA ENSEMBLE - Breaking 90%
Strategies: Beam + MultiBeam + Buddy + Blocks + Random + Reverse
------------------------------------------------------------
0      | 130.16       | PASS
1      | 32.69        | PASS
2      | 63.09        | PASS
3      | 44.77        | PASS
4      | 48.36        | PASS
5      | 65.50        | PASS
6      | 54.38        | PASS
7      | 33.92        | PASS
8      | 56.45        | PASS
9      | 62.20        | PASS
10     | 52.68        | PASS
11     | 63.37        | PASS
12     | 47.52        | PASS
13     | 77.19        | PASS
14     | 117.13       | PASS
15     | 123.04       | PASS
16     | 37.99        | PASS
17     | 51.30        | PASS
18     | 742.78       | PASS
19     | 62.46        | PASS
20     | 79.46        | PASS
21     | 24.20        | PASS
22     | 41.69        | PASS
23     | 60.20        | PASS
24     | 26.16        | PASS
25     | 2316.45      | FAIL
26     | 7895.27      | FAIL
27     | 33.45        | PASS
28     | 41.76        | PAS